In [1]:
import requests
import pandas as pd 
import time
import calendar
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry  
import csv 
# api_key='b0b16ad6834c4ebea1b181e3afa92e21'
api_key='e62c395ec9844095ada4b688d012a975'
technical='time_series'
ticker='BTC/USD'
timezone='Asia/Calcutta'
interval='30min'

In [321]:
data=[]
for yr in range(2022,2026):
    for m in range(1,5):
        last_day=calendar.monthrange(yr,3*m)[1]
        start_date=f'{yr}-{(3*m-2):02d}-01T00:00:00'
        end_date=f'{yr}-{(3*m):02d}-{last_day}T23:59:59'
        api_url=f'https://api.twelvedata.com/{technical}?symbol={ticker}&order=asc&timezone={timezone}&start_date={start_date}&end_date={end_date}&interval={interval}&outputsize=5000&apikey={api_key}'
        while True:
            session=requests.Session()
            retries=Retry(total=5,backoff_factor=1,status_forcelist=[429,500,502,503,504])
            session.mount("https://",HTTPAdapter(max_retries=retries))
            fetch=session.get(api_url).json()
            if "values" in fetch:
                df=pd.DataFrame(fetch['values'])
                data.append(df)
                time.sleep(8)
                break
            elif "code" in fetch and fetch["code"]==429:
                print("Rate limit exceeded")
                time.sleep(15)
            else: 
                print("API Error:",fetch)
                fetch=None
                break
data=pd.concat(data,ignore_index=True)
data['open']=pd.to_numeric(data['open'])
data['close']=pd.to_numeric(data['close'])
data['abs_body']=abs(data['close']-data['open'])
data['body']=(data['close']-data['open'])
data

KeyboardInterrupt: 

In [70]:
data['datetime']=pd.to_datetime(data['datetime'])
missing_times=pd.date_range("2022-01-01", "2025-12-31 23:30", freq="30T").difference(data['datetime'])
missing_times

C:\Users\#samay\AppData\Local\Temp\ipykernel_14444\2112058808.py:2: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  missing_times=pd.date_range("2022-01-01", "2025-12-31 23:30", freq="30T").difference(data['datetime'])


DatetimeIndex(['2022-02-19 19:30:00', '2022-02-19 20:00:00',
               '2022-02-19 20:30:00', '2022-02-19 21:00:00',
               '2022-02-19 21:30:00', '2022-02-19 22:00:00',
               '2022-02-19 22:30:00', '2022-02-19 23:00:00',
               '2022-02-19 23:30:00', '2022-02-20 00:00:00',
               ...
               '2025-10-25 21:30:00', '2025-10-25 22:00:00',
               '2025-10-25 22:30:00', '2025-10-25 23:00:00',
               '2025-10-25 23:30:00', '2025-10-26 00:00:00',
               '2025-10-26 00:30:00', '2025-10-26 01:00:00',
               '2025-10-26 01:30:00', '2025-10-26 02:00:00'],
              dtype='datetime64[ns]', length=131, freq=None)

In [71]:
missing_times=pd.date_range(start="2022-02-19 19:30",end="2022-02-20 02:30",freq="30min")
missing_df=pd.DataFrame({
    "datetime":missing_times,
    "open":[40417.019531, 40382.390620, 40495.820310, 40316.121090, 39856.980470, 40183.230470, 39976.660160, 40073.031250, 40052.289060, 39966.011719, 40165.449220, 40245.160160, 40225.781250, 40079.878910, 39969.539060],
    "high":[40497.32812, 40449.48047, 40739.82031, 40316.12109, 40377.17188, 40382.75, 40360.03125, 40164.83984, 40052.28906, 40194.71875, 40268.60156, 40353.83984, 40294.37891, 40178.96094, 40149.51172],
    "low":[40316.82031, 39673.44922, 40278.39844, 39693.89844, 39464.60938, 40004.41016, 39976.66016, 39963.35156, 39825.62109, 39943.26172, 40084.64062, 40094.92969, 40050.39062, 39960.55859, 39916.57031],
    "close":[40354.148440, 40437.609380, 40278.398440, 39790.531250, 40153.898440, 40004.410160, 40095.351560, 40032.710940, 39983.800780, 40193.539060, 40268.601560, 40218.441410, 40123.281250, 39966.988280, 40128.75],
    "abs_body":[62.871091,55.21876,217.42187,525.58984,296.91797,178.82031,118.6914,40.32031,68.48828,227.527341,103.15234,26.71875,102.5,112.89063,159.21094],
    "body":[-62.871091,55.21876,-217.42187,-525.58984,296.91797,-178.82031,118.6914,-40.32031,-68.48828,227.527341,103.15234,-26.71875,-102.5,-112.89063,159.21094]   
})
data=pd.concat([data,missing_df],ignore_index=True)
data

,datetime,open,high,low,close,abs_body,body
0,2022-01-01 00:00:00,46894.300780,46951.76953,46557.17969,46640.53125,253.769530,-253.769530
1,2022-01-01 00:30:00,46656.261720,46656.26172,46075,46080.87109,575.390630,-575.390630
2,2022-01-01 01:00:00,46064.070312,46318.87109,45656.55078,45693.14844,370.921872,-370.921872
3,2022-01-01 01:30:00,45671.730470,46195.80859,45655.42969,46037.41016,365.679690,365.679690
4,2022-01-01 02:00:00,46042.019531,46115.21094,45823.94922,45881.53906,160.480471,-160.480471
...,...,...,...,...,...,...,...
70007,2022-02-20 00:30:00,40165.449220,40268.60156,40084.64062,40268.60156,103.152340,103.152340
70008,2022-02-20 01:00:00,40245.160160,40353.83984,40094.92969,40218.44141,26.718750,-26.718750
70009,2022-02-20 01:30:00,40225.781250,40294.37891,40050.39062,40123.28125,102.500000,-102.500000
70010,2022-02-20 02:00:00,40079.878910,40178.96094,39960.55859,39966.98828,112.890630,-112.890630


In [72]:
missing_times=pd.date_range(start="2022-04-03 02:00",end="2022-04-03 02:00",freq="30min")
missing_df=pd.DataFrame({
    "datetime":missing_times,
    "open":[46324.46094],
    "high":[46391.42188],
    "low":[46111.011719],
    "close":[46164.92969],
    "abs_body":[159.53125],
    "body":[-159.53125]   
})
data=pd.concat([data,missing_df],ignore_index=True)
data

,datetime,open,high,low,close,abs_body,body
0,2022-01-01 00:00:00,46894.300780,46951.76953,46557.17969,46640.53125,253.769530,-253.769530
1,2022-01-01 00:30:00,46656.261720,46656.26172,46075,46080.87109,575.390630,-575.390630
2,2022-01-01 01:00:00,46064.070312,46318.87109,45656.55078,45693.14844,370.921872,-370.921872
3,2022-01-01 01:30:00,45671.730470,46195.80859,45655.42969,46037.41016,365.679690,365.679690
4,2022-01-01 02:00:00,46042.019531,46115.21094,45823.94922,45881.53906,160.480471,-160.480471
...,...,...,...,...,...,...,...
70008,2022-02-20 01:00:00,40245.160160,40353.83984,40094.92969,40218.44141,26.718750,-26.718750
70009,2022-02-20 01:30:00,40225.781250,40294.37891,40050.39062,40123.28125,102.500000,-102.500000
70010,2022-02-20 02:00:00,40079.878910,40178.96094,39960.55859,39966.98828,112.890630,-112.890630
70011,2022-02-20 02:30:00,39969.539060,40149.51172,39916.57031,40128.75000,159.210940,159.210940


In [73]:
missing_times=pd.date_range(start="2022-05-06 00:30",end="2022-05-06 01:00",freq="30min")
missing_df=pd.DataFrame({
    "datetime":missing_times,
    "open":[39601.14844,39844.64062],
    "high":[39972.55078,39903.19922],
    "low":[39545.058594,39671.89844],
    "close":[39840.53906,39816.12109],
    "abs_body":[239.39062,28.51953],
    "body":[239.39062,-28.51953]   
})
data=pd.concat([data,missing_df],ignore_index=True)
data

,datetime,open,high,low,close,abs_body,body
0,2022-01-01 00:00:00,46894.300780,46951.76953,46557.17969,46640.53125,253.769530,-253.769530
1,2022-01-01 00:30:00,46656.261720,46656.26172,46075,46080.87109,575.390630,-575.390630
2,2022-01-01 01:00:00,46064.070312,46318.87109,45656.55078,45693.14844,370.921872,-370.921872
3,2022-01-01 01:30:00,45671.730470,46195.80859,45655.42969,46037.41016,365.679690,365.679690
4,2022-01-01 02:00:00,46042.019531,46115.21094,45823.94922,45881.53906,160.480471,-160.480471
...,...,...,...,...,...,...,...
70010,2022-02-20 02:00:00,40079.878910,40178.96094,39960.55859,39966.98828,112.890630,-112.890630
70011,2022-02-20 02:30:00,39969.539060,40149.51172,39916.57031,40128.75000,159.210940,159.210940
70012,2022-04-03 02:00:00,46324.460940,46391.42188,46111.011719,46164.92969,159.531250,-159.531250
70013,2022-05-06 00:30:00,39601.148440,39972.55078,39545.058594,39840.53906,239.390620,239.390620


In [74]:
missing_times=pd.date_range(start="2022-10-22 17:30",end="2022-10-22 18:30",freq="30min")
missing_df=pd.DataFrame({
    "datetime":missing_times,
    "open":[18947.509770,18783.619140,18870.109380],
    "high":[18947.50977,18884.25,19004.060547],
    "low":[18679.75,18734.41016,18870.10938],
    "close":[18782.14062,18884.25,18962.26953],
    "abs_body":[165.369150,100.630860,92.16015],
    "body":[-165.369150,100.630860,92.160150]   
})
data=pd.concat([data,missing_df],ignore_index=True)
data

,datetime,open,high,low,close,abs_body,body
0,2022-01-01 00:00:00,46894.300780,46951.76953,46557.17969,46640.53125,253.769530,-253.769530
1,2022-01-01 00:30:00,46656.261720,46656.26172,46075,46080.87109,575.390630,-575.390630
2,2022-01-01 01:00:00,46064.070312,46318.87109,45656.55078,45693.14844,370.921872,-370.921872
3,2022-01-01 01:30:00,45671.730470,46195.80859,45655.42969,46037.41016,365.679690,365.679690
4,2022-01-01 02:00:00,46042.019531,46115.21094,45823.94922,45881.53906,160.480471,-160.480471
...,...,...,...,...,...,...,...
70013,2022-05-06 00:30:00,39601.148440,39972.55078,39545.058594,39840.53906,239.390620,239.390620
70014,2022-05-06 01:00:00,39844.640620,39903.19922,39671.89844,39816.12109,28.519530,-28.519530
70015,2022-10-22 17:30:00,18947.509770,18947.50977,18679.75,18782.14062,165.369150,-165.369150
70016,2022-10-22 18:00:00,18783.619140,18884.25,18734.41016,18884.25000,100.630860,100.630860


In [75]:
missing_times=pd.date_range(start="2022-11-17 04:30",end="2022-11-17 05:30",freq="30min")
missing_df=pd.DataFrame({
    "datetime":missing_times,
    "open":[16861.900390,16844.580080,16880.15039],
    "high":[16877.58984,16882.83008,16911.92969],
    "low":[16824.43945,16820.90039,16782.50977],
    "close":[16839.30078,16880.15039,16782.50977],
    "abs_body":[22.599610,35.570310,97.640620],
    "body":[-22.599610,35.570310,-97.640620]   
})
data=pd.concat([data,missing_df],ignore_index=True)
data

,datetime,open,high,low,close,abs_body,body
0,2022-01-01 00:00:00,46894.300780,46951.76953,46557.17969,46640.53125,253.769530,-253.769530
1,2022-01-01 00:30:00,46656.261720,46656.26172,46075,46080.87109,575.390630,-575.390630
2,2022-01-01 01:00:00,46064.070312,46318.87109,45656.55078,45693.14844,370.921872,-370.921872
3,2022-01-01 01:30:00,45671.730470,46195.80859,45655.42969,46037.41016,365.679690,365.679690
4,2022-01-01 02:00:00,46042.019531,46115.21094,45823.94922,45881.53906,160.480471,-160.480471
...,...,...,...,...,...,...,...
70016,2022-10-22 18:00:00,18783.619140,18884.25,18734.41016,18884.25000,100.630860,100.630860
70017,2022-10-22 18:30:00,18870.109380,19004.060547,18870.10938,18962.26953,92.160150,92.160150
70018,2022-11-17 04:30:00,16861.900390,16877.58984,16824.43945,16839.30078,22.599610,-22.599610
70019,2022-11-17 05:00:00,16844.580080,16882.83008,16820.90039,16880.15039,35.570310,35.570310


In [76]:
missing_times=pd.date_range(start="2022-11-18 05:00",end="2022-11-18 10:00",freq="30min")
missing_df=pd.DataFrame({
    "datetime":missing_times,
    "open":[16844.580080,16880.15039,16681.929690,16688.31055,16697.230470,16684.380860,16653.359380,16584.320310,16580.169920,16508.189450,16470.33008],
    "high":[16882.83008,16911.92969,16711.84961,16727.19922,16710.41992,16691.91016,16672.31055,16593.53906,16582.75977,16509.92969,16519.50977],
    "low":[16820.90039,16782.50977,16660.23047,16681.28906,16671.66992,16642.070312,16582.31055,16568.24023,16498.17969,16403.44922,16467.25],
    "close":[16880.15039,16782.50977,16688.230470,16697.349610,16680.490230,16655.009766,16582.310550,16583.050781,16502.509770,16472.599610,16489.439450],
    "abs_body":[35.570310,97.640620,6.30078,9.03906,16.74024,29.371094,71.04883,1.269529,77.660150,35.589840,19.109370],
    "body":[35.570310,-97.640620,6.300780,9.039060,-16.740240,-29.371094,-71.048830,-1.269529,-77.660150,-35.589840,19.109370]   
})
data=pd.concat([data,missing_df],ignore_index=True)
data

,datetime,open,high,low,close,abs_body,body
0,2022-01-01 00:00:00,46894.300780,46951.76953,46557.17969,46640.531250,253.769530,-253.769530
1,2022-01-01 00:30:00,46656.261720,46656.26172,46075,46080.871090,575.390630,-575.390630
2,2022-01-01 01:00:00,46064.070312,46318.87109,45656.55078,45693.148440,370.921872,-370.921872
3,2022-01-01 01:30:00,45671.730470,46195.80859,45655.42969,46037.410160,365.679690,365.679690
4,2022-01-01 02:00:00,46042.019531,46115.21094,45823.94922,45881.539060,160.480471,-160.480471
...,...,...,...,...,...,...,...
70027,2022-11-18 08:00:00,16653.359380,16672.31055,16582.31055,16582.310550,71.048830,-71.048830
70028,2022-11-18 08:30:00,16584.320310,16593.53906,16568.24023,16583.050781,1.269529,-1.269529
70029,2022-11-18 09:00:00,16580.169920,16582.75977,16498.17969,16502.509770,77.660150,-77.660150
70030,2022-11-18 09:30:00,16508.189450,16509.92969,16403.44922,16472.599610,35.589840,-35.589840


In [77]:
missing_times=pd.date_range(start="2022-11-19 05:00",end="2022-11-19 19:30",freq="30min")
missing_df=pd.DataFrame({
    "datetime":missing_times,
    "open":[16844.58008,16880.15039,16681.92969,16688.31055,16697.23047,16684.38086,16653.35938,16584.32031,16580.16992,16508.18945,16470.33008,16825.42969,16810.34961,16791.14062,16744.73047,16778.31055,16767.76953,16727.86914,16711.51953,16715.65039,16727.16992,16730.38086,16781.96094,16750.039062,16736.25977,16737.16016,16754.5293,16759.91992,16777.41992,16742.2793],
    "high":[16882.83008,16911.92969,16711.84961,16727.19922,16710.41992,16691.91016,16672.31055,16593.53906,16582.75977,16509.92969,16519.50977,16825.42969,16815.41016,16806.4707,16788.050781,16785.25977,16770.11914,16731.35938,16721.9707,16740.089844,16745.68945,16789,16833.86914,16769.050781,16746.17969,16764.41992,16771.84961,16798.17969,16796.070312,16765.97070],
    "low":[16820.90039,16782.50977,16660.23047,16681.28906,16671.66992,16642.070312,16582.31055,16568.24023,16498.17969,16403.44922,16467.25,16806.82031,16772.98047,16744.21094,16743.75,16751.63086,16708.80078,16663.88086,16688.91016,16711.58984,16707.10938,16726.96094,16708.39062,16726.63086,16702.25977,16730.90039,16745.53906,16759.91992,16724.74023,16730.83008],
    "close":[16880.15039,16782.50977,16688.23047,16697.34961,16680.49023,16655.009766,16582.31055,16583.050781,16502.50977,16472.59961,16489.43945,16806.82031,16789.84961,16749.4707,16778.58984,16770.11914,16731.13086,16712.51953,16714.67969,16728.48047,16729.35938,16775.28906,16756.53906,16737.25,16737.83984,16753.91992,16764.2207,16777.070312,16744.66016,16759.94922],
    "abs_body":[35.57031,97.64062,6.30078,9.03906,16.74024,29.371094,71.04883,1.269529,77.66015,35.58984,19.10937,18.609380,20.5,41.66992,33.85937,8.19141,36.63867,15.34961,3.16016,12.83008,2.18946,44.9082,25.42188,12.789062,1.58007,16.75976,9.6914,17.150392,32.75976,17.66992],
    "body":[35.57031,-97.64062,6.30078,9.03906,-16.74024,-29.371094,-71.04883,-1.269529,-77.66015,-35.58984,19.10937,-18.609380,-20.5,-41.66992,33.85937,-8.19141,-36.63867,-15.34961,3.16016,12.83008,2.18946,44.9082,-25.42188,-12.789062,1.58007,16.75976,9.6914,17.150392,-32.75976,17.66992]   
})
data=pd.concat([data,missing_df],ignore_index=True)
data

,datetime,open,high,low,close,abs_body,body
0,2022-01-01 00:00:00,46894.300780,46951.76953,46557.17969,46640.531250,253.769530,-253.769530
1,2022-01-01 00:30:00,46656.261720,46656.26172,46075,46080.871090,575.390630,-575.390630
2,2022-01-01 01:00:00,46064.070312,46318.87109,45656.55078,45693.148440,370.921872,-370.921872
3,2022-01-01 01:30:00,45671.730470,46195.80859,45655.42969,46037.410160,365.679690,365.679690
4,2022-01-01 02:00:00,46042.019531,46115.21094,45823.94922,45881.539060,160.480471,-160.480471
...,...,...,...,...,...,...,...
70057,2022-11-19 17:30:00,16737.160160,16764.41992,16730.90039,16753.919920,16.759760,16.759760
70058,2022-11-19 18:00:00,16754.529300,16771.84961,16745.53906,16764.220700,9.691400,9.691400
70059,2022-11-19 18:30:00,16759.919920,16798.17969,16759.91992,16777.070312,17.150392,17.150392
70060,2022-11-19 19:00:00,16777.419920,16796.070312,16724.74023,16744.660160,32.759760,-32.759760


In [78]:
missing_times=pd.date_range(start="2022-11-20 14:30",end="2022-11-20 15:00",freq="30min")
missing_df=pd.DataFrame({
    "datetime":missing_times,
    "open":[16715.65039,16727.16992],
    "high":[16740.089844,16745.68945],
    "low":[16711.58984,16707.10938],
    "close":[16728.48047,16729.35938],
    "abs_body":[12.83008,2.18946],
    "body":[12.83008,2.18946]   
})
data=pd.concat([data,missing_df],ignore_index=True)
data

,datetime,open,high,low,close,abs_body,body
0,2022-01-01 00:00:00,46894.300780,46951.76953,46557.17969,46640.531250,253.769530,-253.769530
1,2022-01-01 00:30:00,46656.261720,46656.26172,46075,46080.871090,575.390630,-575.390630
2,2022-01-01 01:00:00,46064.070312,46318.87109,45656.55078,45693.148440,370.921872,-370.921872
3,2022-01-01 01:30:00,45671.730470,46195.80859,45655.42969,46037.410160,365.679690,365.679690
4,2022-01-01 02:00:00,46042.019531,46115.21094,45823.94922,45881.539060,160.480471,-160.480471
...,...,...,...,...,...,...,...
70059,2022-11-19 18:30:00,16759.919920,16798.17969,16759.91992,16777.070312,17.150392,17.150392
70060,2022-11-19 19:00:00,16777.419920,16796.070312,16724.74023,16744.660160,32.759760,-32.759760
70061,2022-11-19 19:30:00,16742.279300,16765.9707,16730.83008,16759.949220,17.669920,17.669920
70062,2022-11-20 14:30:00,16715.650390,16740.089844,16711.58984,16728.480470,12.830080,12.830080


In [79]:
missing_times=pd.date_range(start="2022-11-20 16:00",end="2022-11-20 16:00",freq="30min")
missing_df=pd.DataFrame({
    "datetime":missing_times,
    "open":[16730.38086],
    "high":[16789],
    "low":[16726.96094],
    "close":[16775.28906],
    "abs_body":[44.9082],
    "body":[44.9082]   
})
data=pd.concat([data,missing_df],ignore_index=True)
data

,datetime,open,high,low,close,abs_body,body
0,2022-01-01 00:00:00,46894.300780,46951.76953,46557.17969,46640.53125,253.769530,-253.769530
1,2022-01-01 00:30:00,46656.261720,46656.26172,46075,46080.87109,575.390630,-575.390630
2,2022-01-01 01:00:00,46064.070312,46318.87109,45656.55078,45693.14844,370.921872,-370.921872
3,2022-01-01 01:30:00,45671.730470,46195.80859,45655.42969,46037.41016,365.679690,365.679690
4,2022-01-01 02:00:00,46042.019531,46115.21094,45823.94922,45881.53906,160.480471,-160.480471
...,...,...,...,...,...,...,...
70060,2022-11-19 19:00:00,16777.419920,16796.070312,16724.74023,16744.66016,32.759760,-32.759760
70061,2022-11-19 19:30:00,16742.279300,16765.9707,16730.83008,16759.94922,17.669920,17.669920
70062,2022-11-20 14:30:00,16715.650390,16740.089844,16711.58984,16728.48047,12.830080,12.830080
70063,2022-11-20 15:00:00,16727.169920,16745.68945,16707.10938,16729.35938,2.189460,2.189460


In [80]:
missing_times=pd.date_range(start="2022-12-03 13:00",end="2022-12-03 16:30",freq="30min")
missing_df=pd.DataFrame({
    "datetime":missing_times,
    "open":[16932.40039,16958.76953,16948.029297,16967.91016,16965.92969,16967.32031,16954.35938,16967.060547],
    "high":[16970.65039,16965.75977,16972.85938,16978.36914,16988.4707,16977.070312,16968.7207,16995.019531],
    "low":[16932.40039,16941.59961,16948.029297,16960.82031,16964.58984,16952.78906,16951.17969,16959.36914],
    "close":[16965.30078,16947.4707,16966.25977,16966.46094,16966.029297,16954.82031,16966.88086,16974.080078],
    "abs_body":[32.90039,11.29883,18.230473,1.44922,0.099607,12.5,12.52148,7.019531],
    "body":[32.90039,-11.29883,-18.230473,-1.44922,0.099607,-12.5,12.52148,7.019531]   
})
data=pd.concat([data,missing_df],ignore_index=True)
data

,datetime,open,high,low,close,abs_body,body
0,2022-01-01 00:00:00,46894.300780,46951.76953,46557.17969,46640.531250,253.769530,-253.769530
1,2022-01-01 00:30:00,46656.261720,46656.26172,46075,46080.871090,575.390630,-575.390630
2,2022-01-01 01:00:00,46064.070312,46318.87109,45656.55078,45693.148440,370.921872,-370.921872
3,2022-01-01 01:30:00,45671.730470,46195.80859,45655.42969,46037.410160,365.679690,365.679690
4,2022-01-01 02:00:00,46042.019531,46115.21094,45823.94922,45881.539060,160.480471,-160.480471
...,...,...,...,...,...,...,...
70068,2022-12-03 14:30:00,16967.910160,16978.36914,16960.82031,16966.460940,1.449220,-1.449220
70069,2022-12-03 15:00:00,16965.929690,16988.4707,16964.58984,16966.029297,0.099607,0.099607
70070,2022-12-03 15:30:00,16967.320310,16977.070312,16952.78906,16954.820310,12.500000,-12.500000
70071,2022-12-03 16:00:00,16954.359380,16968.7207,16951.17969,16966.880860,12.521480,12.521480


In [81]:
missing_times=pd.date_range(start="2023-02-22 21:30",end="2023-02-23 11:30",freq="30min")
missing_df=pd.DataFrame({
    "datetime":missing_times,
    "open":[24563.85938,24381.33984,24409.80078,24401.59961,24641.91016,24629.23047,24689.48047,24640.019531,24607.7207,24449.66992,24459.94922,24241.070312,24190.16992,24295.82031,24389,24361.81055,24452.98047,24412.73047,24403.23047,24360.14062,24203.83984,24036.24023,24172.31055,24229.11914,24201.96094,24163.82031,24158.83984,24026.33008,24102.78906],
    "high":[24587.75977,24444.42969,24444.65039,24652.85938,24746.90039,24732.019531,24693.90039,24686.53906,24642.88086,24525.67969,24536.65039,24332.26953,24316.30078,24428.84961,24404.33984,24451.90039,24474.51953,24425.5293,24432.75,24392.070312,24203.83984,24195.30078,24270.40039,24247.11914,24222.73047,24170.32031,24168.15039,24113.029297,24128.25977],
    "low":[24321.32031,24301.5293,24380.76953,24401.59961,24589.96094,24569.58008,24564.4707,24558.17969,24450.56055,24434.57031,24159.85938,24173.41016,24174.28906,24280.82031,24341.019531,24351.32031,24411.49023,24276.41016,24356.019531,24156.5,23969.66016,23873.0097656,24133.11914,24155.86914,24160.029297,24031.099609,24007.91992,24013.31055,24028.17969],
    "close":[24377.19922,24403.060547,24412.69922,24652.85938,24631.66016,24693.90039,24636.63086,24595.96094,24450.88086,24464.41016,24257.78906,24175.57031,24295.33984,24384.94922,24356.9707,24451.90039,24411.83984,24402.7793,24365.51953,24195.76953,24036.60938,24169.50977,24238.68945,24200.099609,24162.17969,24161.21094,24032.81055,24107.34961,24044.48047],
    "abs_body":[186.66016,21.720707,2.89844,251.25977,10.25,64.66992,52.84961,44.058591,156.83984,14.74024,202.16016,65.500002,105.16992,89.12891,32.0293,90.08984,41.14063,9.95117,37.71094,164.37109,167.23046,133.26954,66.3789,29.019531,39.78125,2.60937,126.02929,81.01953,58.30859],
    "body":[-186.66016,21.720707,2.89844,251.25977,-10.25,64.66992,-52.84961,-44.058591,-156.83984,14.74024,-202.16016,-65.500002,105.16992,89.12891,-32.0293,90.08984,-41.14063,-9.95117,-37.71094,-164.37109,-167.23046,133.26954,66.3789,-29.019531,-39.78125,-2.60937,-126.02929,81.01953,-58.30859]   
})
data=pd.concat([data,missing_df],ignore_index=True)
data

,datetime,open,high,low,close,abs_body,body
0,2022-01-01 00:00:00,46894.300780,46951.76953,46557.17969,46640.53125,253.769530,-253.769530
1,2022-01-01 00:30:00,46656.261720,46656.26172,46075,46080.87109,575.390630,-575.390630
2,2022-01-01 01:00:00,46064.070312,46318.87109,45656.55078,45693.14844,370.921872,-370.921872
3,2022-01-01 01:30:00,45671.730470,46195.80859,45655.42969,46037.41016,365.679690,365.679690
4,2022-01-01 02:00:00,46042.019531,46115.21094,45823.94922,45881.53906,160.480471,-160.480471
...,...,...,...,...,...,...,...
70097,2023-02-23 09:30:00,24201.960940,24222.73047,24160.029297,24162.17969,39.781250,-39.781250
70098,2023-02-23 10:00:00,24163.820310,24170.32031,24031.099609,24161.21094,2.609370,-2.609370
70099,2023-02-23 10:30:00,24158.839840,24168.15039,24007.91992,24032.81055,126.029290,-126.029290
70100,2023-02-23 11:00:00,24026.330080,24113.029297,24013.31055,24107.34961,81.019530,81.019530


In [82]:
missing_times=pd.date_range(start="2023-03-04 23:00",end="2023-03-05 01:30",freq="30min")
missing_df=pd.DataFrame({
    "datetime":missing_times,
    "open":[22422.019531,22420.60938,22386.86914,22314.14062,22306.85938,22354.26953],
    "high":[22448.36914,22422.050781,22394.24023,22355.99023,22389.84961,22377.71094],
    "low":[22395.92969,22375.25,22263.57031,22294.13086,22301.080078,22333.2207],
    "close":[22417.42969,22394.24023,22316.21094,22311.89062,22356.48047,22355.060547],
    "abs_body":[4.589841,26.36915,70.6582,2.25,49.62109,0.791017],
    "body":[-4.589841,-26.36915,-70.6582,-2.25,49.62109,0.791017]
})
data=pd.concat([data,missing_df],ignore_index=True)
data

,datetime,open,high,low,close,abs_body,body
0,2022-01-01 00:00:00,46894.300780,46951.76953,46557.17969,46640.531250,253.769530,-253.769530
1,2022-01-01 00:30:00,46656.261720,46656.26172,46075,46080.871090,575.390630,-575.390630
2,2022-01-01 01:00:00,46064.070312,46318.87109,45656.55078,45693.148440,370.921872,-370.921872
3,2022-01-01 01:30:00,45671.730470,46195.80859,45655.42969,46037.410160,365.679690,365.679690
4,2022-01-01 02:00:00,46042.019531,46115.21094,45823.94922,45881.539060,160.480471,-160.480471
...,...,...,...,...,...,...,...
70103,2023-03-04 23:30:00,22420.609380,22422.050781,22375.25,22394.240230,26.369150,-26.369150
70104,2023-03-05 00:00:00,22386.869140,22394.24023,22263.57031,22316.210940,70.658200,-70.658200
70105,2023-03-05 00:30:00,22314.140620,22355.99023,22294.13086,22311.890620,2.250000,-2.250000
70106,2023-03-05 01:00:00,22306.859380,22389.84961,22301.080078,22356.480470,49.621090,49.621090


In [83]:
missing_times=pd.date_range(start="2023-07-20 00:00",end="2023-07-20 03:30",freq="30min")
missing_df=pd.DataFrame({
    "datetime":missing_times,
    "open":[29929.14,29909.63,29775.01,29720.64,29728.11,29783.77,29830.69,29807.85],
    "high":[29932.4,29946.23,29817.33,29794.65,29818.44,29841.05,29831.53,29823.14],
    "low":[29859.69,29771.55,29685.27,29704.95,29690.39,29756.01,29778.89,29768.22],
    "close":[29910.96,29775,29722.14,29728.11,29783.61,29830.7,29807.88,29790.82],
    "abs_body":[18.18,134.63,52.87,7.47,55.5,46.93,22.81,17.03],
    "body":[-18.18,-134.63,-52.87,7.47,55.5,46.93,-22.81,-17.03]
})
data=pd.concat([data,missing_df],ignore_index=True)
data

,datetime,open,high,low,close,abs_body,body
0,2022-01-01 00:00:00,46894.300780,46951.76953,46557.17969,46640.53125,253.769530,-253.769530
1,2022-01-01 00:30:00,46656.261720,46656.26172,46075,46080.87109,575.390630,-575.390630
2,2022-01-01 01:00:00,46064.070312,46318.87109,45656.55078,45693.14844,370.921872,-370.921872
3,2022-01-01 01:30:00,45671.730470,46195.80859,45655.42969,46037.41016,365.679690,365.679690
4,2022-01-01 02:00:00,46042.019531,46115.21094,45823.94922,45881.53906,160.480471,-160.480471
...,...,...,...,...,...,...,...
70111,2023-07-20 01:30:00,29720.640000,29794.65,29704.95,29728.11000,7.470000,7.470000
70112,2023-07-20 02:00:00,29728.110000,29818.44,29690.39,29783.61000,55.500000,55.500000
70113,2023-07-20 02:30:00,29783.770000,29841.05,29756.01,29830.70000,46.930000,46.930000
70114,2023-07-20 03:00:00,29830.690000,29831.53,29778.89,29807.88000,22.810000,-22.810000


In [84]:
missing_times=pd.date_range(start="2024-10-26 22:00",end="2024-10-26 22:00",freq="30min")
missing_df=pd.DataFrame({
    "datetime":missing_times,
    "open":[67579.54],
    "high":[67864.16],
    "low":[67512.9],
    "close":[67633.74],
    "abs_body":[54.2],
    "body":[54.2]
})
data=pd.concat([data,missing_df],ignore_index=True)
data

,datetime,open,high,low,close,abs_body,body
0,2022-01-01 00:00:00,46894.300780,46951.76953,46557.17969,46640.53125,253.769530,-253.769530
1,2022-01-01 00:30:00,46656.261720,46656.26172,46075,46080.87109,575.390630,-575.390630
2,2022-01-01 01:00:00,46064.070312,46318.87109,45656.55078,45693.14844,370.921872,-370.921872
3,2022-01-01 01:30:00,45671.730470,46195.80859,45655.42969,46037.41016,365.679690,365.679690
4,2022-01-01 02:00:00,46042.019531,46115.21094,45823.94922,45881.53906,160.480471,-160.480471
...,...,...,...,...,...,...,...
70112,2023-07-20 02:00:00,29728.110000,29818.44,29690.39,29783.61000,55.500000,55.500000
70113,2023-07-20 02:30:00,29783.770000,29841.05,29756.01,29830.70000,46.930000,46.930000
70114,2023-07-20 03:00:00,29830.690000,29831.53,29778.89,29807.88000,22.810000,-22.810000
70115,2023-07-20 03:30:00,29807.850000,29823.14,29768.22,29790.82000,17.030000,-17.030000


In [85]:
missing_times=pd.date_range(start="2025-10-25 21:00",end="2025-10-26 02:00",freq="30min")
missing_df=pd.DataFrame({
    "datetime":missing_times,
    "open":[110608.48,110070.08,110370.87,110282.04,110516.01,110231.46,110528.01,110621.56,110948.65,110624.01,110754],
    "high":[110636.73,110411.24,110378.96,110600,110519.16,110658.51,110654.51,110980.7,110955.64,110817.37,111040.55],
    "low":[109910.01,109804.93,109930,110195.8,110231.47,110165.84,110422,110617.05,110506.46,110485.4,110676.22],
    "close":[110070.09,110374.01,110286.93,110516.01,110233.73,110524.36,110621.56,110948.65,110624.14,110753.99,110920.22],
    "abs_body":[538.39,303.93,83.94,233.97,282.28,292.9,93.55,327.09,324.51,129.98,166.22],
    "body":[-538.39,303.93,-83.94,233.97,-282.28,292.9,93.55,327.09,-324.51,129.98,166.22]
})
data=pd.concat([data,missing_df],ignore_index=True)
data

,datetime,open,high,low,close,abs_body,body
0,2022-01-01 00:00:00,46894.300780,46951.76953,46557.17969,46640.53125,253.769530,-253.769530
1,2022-01-01 00:30:00,46656.261720,46656.26172,46075,46080.87109,575.390630,-575.390630
2,2022-01-01 01:00:00,46064.070312,46318.87109,45656.55078,45693.14844,370.921872,-370.921872
3,2022-01-01 01:30:00,45671.730470,46195.80859,45655.42969,46037.41016,365.679690,365.679690
4,2022-01-01 02:00:00,46042.019531,46115.21094,45823.94922,45881.53906,160.480471,-160.480471
...,...,...,...,...,...,...,...
70123,2025-10-26 00:00:00,110528.010000,110654.51,110422.0,110621.56000,93.550000,93.550000
70124,2025-10-26 00:30:00,110621.560000,110980.7,110617.05,110948.65000,327.090000,327.090000
70125,2025-10-26 01:00:00,110948.650000,110955.64,110506.46,110624.14000,324.510000,-324.510000
70126,2025-10-26 01:30:00,110624.010000,110817.37,110485.4,110753.99000,129.980000,129.980000


In [228]:
data['datetime']=pd.to_datetime(data['datetime'])
data=data.sort_values("datetime").reset_index(drop=True)
data

,datetime,open,high,low,close,abs_body,body
0,2022-01-01 00:00:00,46894.300780,46951.76953,46557.17969,46640.53125,253.769530,-253.769530
1,2022-01-01 00:30:00,46656.261720,46656.26172,46075,46080.87109,575.390630,-575.390630
2,2022-01-01 01:00:00,46064.070312,46318.87109,45656.55078,45693.14844,370.921872,-370.921872
3,2022-01-01 01:30:00,45671.730470,46195.80859,45655.42969,46037.41016,365.679690,365.679690
4,2022-01-01 02:00:00,46042.019531,46115.21094,45823.94922,45881.53906,160.480471,-160.480471
...,...,...,...,...,...,...,...
70123,2025-12-31 21:30:00,87830.030000,87932.17,87558,87792.55000,37.480000,-37.480000
70124,2025-12-31 22:00:00,87797.910000,87813.02,87259.01,87531.06000,266.850000,-266.850000
70125,2025-12-31 22:30:00,87531.060000,87637.99,87356.29,87496.24000,34.820000,-34.820000
70126,2025-12-31 23:00:00,87496.250000,87550.34,87400.76,87487.96000,8.290000,-8.290000


In [171]:
df_volume=[]
df_takervolume=[]
df_opentime=[]
df_trades=[]
interval='30m'
for yr in range(2022,2026):
    for m in range(1,13): 
        start_date=f'{yr}-{(m):02d}-01'
        end_date=f'{yr}-{(m):02d}-16'
        start_ms=int(time.mktime(time.strptime(start_date,"%Y-%m-%d")))*1000
        end_ms=int(time.mktime(time.strptime(end_date,"%Y-%m-%d")))*1000
        api_url=f"https://api1.binance.com/api/v3/klines?symbol=BTCUSDT&interval={interval}&startTime={start_ms}&endTime={end_ms}&limit=1000"
        while True:
            session=requests.Session()
            retries=Retry(total=5,backoff_factor=1,status_forcelist=[429,500,502,503,504])
            session.mount("https://",HTTPAdapter(max_retries=retries))
            fetch=session.get(api_url).json()
            if fetch:
                open_time=[value[0] for value in fetch]
                volume=[float(value[7]) for value in fetch]
                taker_volume=[float(value[10]) for value in fetch]
                trades=[float(value[8]) for value in fetch]
                df_volume.extend(volume)
                df_opentime.extend(open_time) 
                df_takervolume.extend(taker_volume)
                df_trades.extend(trades)
                time.sleep(4)
                break
            else: 
                print("API Error:",fetch)
                fetch=None
                break
        last_day=calendar.monthrange(yr,m)[1]
        start_date=f'{yr}-{(m):02d}-16'
        end_date=f'{yr}-{(m):02d}-{(last_day)}'
        start_ms=int(time.mktime(time.strptime(start_date,"%Y-%m-%d")))*1000
        end_ms=int(time.mktime(time.strptime(end_date,"%Y-%m-%d")))*1000
        api_url=f"https://api1.binance.com/api/v3/klines?symbol=BTCUSDT&interval={interval}&startTime={start_ms}&endTime={end_ms}&limit=1000"
        while True:
            session=requests.Session()
            retries=Retry(total=5,backoff_factor=1,status_forcelist=[429,500,502,503,504])
            session.mount("https://",HTTPAdapter(max_retries=retries))
            fetch=session.get(api_url).json()
            if fetch:
                open_time=[value[0] for value in fetch]
                volume=[float(value[7]) for value in fetch]
                taker_volume=[float(value[10]) for value in fetch]
                trades=[float(value[8]) for value in fetch]
                df_volume.extend(volume)
                df_opentime.extend(open_time) 
                df_takervolume.extend(taker_volume)
                df_trades.extend(trades)
                time.sleep(4)
                break
            else: 
                print("API Error:",fetch)
                fetch=None
                break
df=pd.DataFrame({
    'open_time':pd.to_datetime(df_opentime,unit='ms'),
    'volume':df_volume,
    'taker_volume':df_takervolume,
    'no_of_trades':df_trades
}).dropna()
df

,open_time,volume,taker_volume,no_of_trades
0,2021-12-31 18:30:00,5.741317e+07,2.218630e+07,33022.0
1,2021-12-31 19:00:00,6.294230e+07,2.873560e+07,37711.0
2,2021-12-31 19:30:00,1.146748e+08,4.930189e+07,49761.0
3,2021-12-31 20:00:00,6.113717e+07,3.375046e+07,32543.0
4,2021-12-31 20:30:00,2.308425e+07,1.213224e+07,18757.0
...,...,...,...,...
67913,2025-12-30 16:30:00,4.550781e+07,1.735067e+07,145135.0
67914,2025-12-30 17:00:00,2.063476e+07,1.072612e+07,102346.0
67915,2025-12-30 17:30:00,4.250846e+07,2.167012e+07,105824.0
67916,2025-12-30 18:00:00,1.873078e+07,1.009447e+07,72103.0


In [227]:
df['open_time']=pd.to_datetime(df['open_time'])
missing_times=pd.date_range("2022-01-01","2025-12-31 23:30",freq="30T").difference(df['open_time'])
missing_times

C:\Users\#samay\AppData\Local\Temp\ipykernel_14444\4086173269.py:2: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  missing_times=pd.date_range("2022-01-01","2025-12-31 23:30",freq="30T").difference(df['open_time'])


DatetimeIndex(['2022-01-30 19:00:00', '2022-01-30 19:30:00',
               '2022-01-30 20:00:00', '2022-01-30 20:30:00',
               '2022-01-30 21:00:00', '2022-01-30 21:30:00',
               '2022-01-30 22:00:00', '2022-01-30 22:30:00',
               '2022-01-30 23:00:00', '2022-01-30 23:30:00',
               ...
               '2025-12-31 19:00:00', '2025-12-31 19:30:00',
               '2025-12-31 20:00:00', '2025-12-31 20:30:00',
               '2025-12-31 21:00:00', '2025-12-31 21:30:00',
               '2025-12-31 22:00:00', '2025-12-31 22:30:00',
               '2025-12-31 23:00:00', '2025-12-31 23:30:00'],
              dtype='datetime64[ns]', length=2269, freq=None)

In [266]:
df['open_time']=pd.to_datetime(df['open_time'],unit='ns',errors='coerce')
df

,open_time,volume,taker_volume,no_of_trades
open_time,,,,
2022-01-01 00:00:00,2022-01-01 00:00:00,3.275119e+07,1.800036e+07,19674.0
2022-01-01 00:30:00,2022-01-01 00:30:00,3.712879e+07,1.946179e+07,18934.0
2022-01-01 01:00:00,2022-01-01 01:00:00,2.754998e+07,1.554390e+07,18930.0
2022-01-01 01:30:00,2022-01-01 01:30:00,1.657717e+07,7.419374e+06,12942.0
2022-01-01 02:00:00,2022-01-01 02:00:00,1.281610e+07,5.467496e+06,13394.0
...,...,...,...,...
2025-12-30 16:30:00,2025-12-30 16:30:00,4.550781e+07,1.735067e+07,145135.0
2025-12-30 17:00:00,2025-12-30 17:00:00,2.063476e+07,1.072612e+07,102346.0
2025-12-30 17:30:00,2025-12-30 17:30:00,4.250846e+07,2.167012e+07,105824.0


In [267]:
df=df[~df.index.duplicated(keep='first')] 
df

,open_time,volume,taker_volume,no_of_trades
open_time,,,,
2022-01-01 00:00:00,2022-01-01 00:00:00,3.275119e+07,1.800036e+07,19674.0
2022-01-01 00:30:00,2022-01-01 00:30:00,3.712879e+07,1.946179e+07,18934.0
2022-01-01 01:00:00,2022-01-01 01:00:00,2.754998e+07,1.554390e+07,18930.0
2022-01-01 01:30:00,2022-01-01 01:30:00,1.657717e+07,7.419374e+06,12942.0
2022-01-01 02:00:00,2022-01-01 02:00:00,1.281610e+07,5.467496e+06,13394.0
...,...,...,...,...
2025-12-30 16:30:00,2025-12-30 16:30:00,4.550781e+07,1.735067e+07,145135.0
2025-12-30 17:00:00,2025-12-30 17:00:00,2.063476e+07,1.072612e+07,102346.0
2025-12-30 17:30:00,2025-12-30 17:30:00,4.250846e+07,2.167012e+07,105824.0


In [268]:
df=df[(df.index>='2022-01-01')&(df.index<='2026-01-01')]
df

,open_time,volume,taker_volume,no_of_trades
open_time,,,,
2022-01-01 00:00:00,2022-01-01 00:00:00,3.275119e+07,1.800036e+07,19674.0
2022-01-01 00:30:00,2022-01-01 00:30:00,3.712879e+07,1.946179e+07,18934.0
2022-01-01 01:00:00,2022-01-01 01:00:00,2.754998e+07,1.554390e+07,18930.0
2022-01-01 01:30:00,2022-01-01 01:30:00,1.657717e+07,7.419374e+06,12942.0
2022-01-01 02:00:00,2022-01-01 02:00:00,1.281610e+07,5.467496e+06,13394.0
...,...,...,...,...
2025-12-30 16:30:00,2025-12-30 16:30:00,4.550781e+07,1.735067e+07,145135.0
2025-12-30 17:00:00,2025-12-30 17:00:00,2.063476e+07,1.072612e+07,102346.0
2025-12-30 17:30:00,2025-12-30 17:30:00,4.250846e+07,2.167012e+07,105824.0


In [269]:
df.index.max()

Timestamp('2025-12-30 18:30:00')

In [270]:
full_range=pd.date_range(start='2022-01-01T00:00:00',end='2025-12-31T23:30:00',freq='30min')
df2=df.reindex(full_range).ffill()
df2=df2.bfill()
df2

,open_time,volume,taker_volume,no_of_trades
2022-01-01 00:00:00,2022-01-01 00:00:00,3.275119e+07,1.800036e+07,19674.0
2022-01-01 00:30:00,2022-01-01 00:30:00,3.712879e+07,1.946179e+07,18934.0
2022-01-01 01:00:00,2022-01-01 01:00:00,2.754998e+07,1.554390e+07,18930.0
2022-01-01 01:30:00,2022-01-01 01:30:00,1.657717e+07,7.419374e+06,12942.0
2022-01-01 02:00:00,2022-01-01 02:00:00,1.281610e+07,5.467496e+06,13394.0
...,...,...,...,...
2025-12-31 21:30:00,2025-12-30 18:30:00,1.963927e+07,9.808946e+06,63776.0
2025-12-31 22:00:00,2025-12-30 18:30:00,1.963927e+07,9.808946e+06,63776.0
2025-12-31 22:30:00,2025-12-30 18:30:00,1.963927e+07,9.808946e+06,63776.0
2025-12-31 23:00:00,2025-12-30 18:30:00,1.963927e+07,9.808946e+06,63776.0


In [286]:
data.index=pd.to_datetime(data['datetime'],unit='ns',errors='coerce')
data

,datetime,open,high,low,close,abs_body,body
datetime,,,,,,,
2022-01-01 00:00:00,2022-01-01 00:00:00,46894.300780,46951.76953,46557.17969,46640.53125,253.769530,-253.769530
2022-01-01 00:30:00,2022-01-01 00:30:00,46656.261720,46656.26172,46075,46080.87109,575.390630,-575.390630
2022-01-01 01:00:00,2022-01-01 01:00:00,46064.070312,46318.87109,45656.55078,45693.14844,370.921872,-370.921872
2022-01-01 01:30:00,2022-01-01 01:30:00,45671.730470,46195.80859,45655.42969,46037.41016,365.679690,365.679690
2022-01-01 02:00:00,2022-01-01 02:00:00,46042.019531,46115.21094,45823.94922,45881.53906,160.480471,-160.480471
...,...,...,...,...,...,...,...
2025-12-31 21:30:00,2025-12-31 21:30:00,87830.030000,87932.17,87558,87792.55000,37.480000,-37.480000
2025-12-31 22:00:00,2025-12-31 22:00:00,87797.910000,87813.02,87259.01,87531.06000,266.850000,-266.850000
2025-12-31 22:30:00,2025-12-31 22:30:00,87531.060000,87637.99,87356.29,87496.24000,34.820000,-34.820000


In [287]:
data.index.min(),data.index.max()

(Timestamp('2022-01-01 00:00:00'), Timestamp('2025-12-31 23:30:00'))

In [288]:
data.index=data.index.astype('int64')
df2.index=df2.index.astype('int64')

In [289]:
merged_data=pd.merge_asof(data,df2,left_index=True,right_index=True,direction='nearest',tolerance=30)
merged_data

,datetime,open,high,low,close,abs_body,body,open_time,volume,taker_volume,no_of_trades
datetime,,,,,,,,,,,
1640995200000000000,2022-01-01 00:00:00,46894.300780,46951.76953,46557.17969,46640.53125,253.769530,-253.769530,2022-01-01 00:00:00,3.275119e+07,1.800036e+07,19674.0
1640997000000000000,2022-01-01 00:30:00,46656.261720,46656.26172,46075,46080.87109,575.390630,-575.390630,2022-01-01 00:30:00,3.712879e+07,1.946179e+07,18934.0
1640998800000000000,2022-01-01 01:00:00,46064.070312,46318.87109,45656.55078,45693.14844,370.921872,-370.921872,2022-01-01 01:00:00,2.754998e+07,1.554390e+07,18930.0
1641000600000000000,2022-01-01 01:30:00,45671.730470,46195.80859,45655.42969,46037.41016,365.679690,365.679690,2022-01-01 01:30:00,1.657717e+07,7.419374e+06,12942.0
1641002400000000000,2022-01-01 02:00:00,46042.019531,46115.21094,45823.94922,45881.53906,160.480471,-160.480471,2022-01-01 02:00:00,1.281610e+07,5.467496e+06,13394.0
...,...,...,...,...,...,...,...,...,...,...,...
1767216600000000000,2025-12-31 21:30:00,87830.030000,87932.17,87558,87792.55000,37.480000,-37.480000,2025-12-30 18:30:00,1.963927e+07,9.808946e+06,63776.0
1767218400000000000,2025-12-31 22:00:00,87797.910000,87813.02,87259.01,87531.06000,266.850000,-266.850000,2025-12-30 18:30:00,1.963927e+07,9.808946e+06,63776.0
1767220200000000000,2025-12-31 22:30:00,87531.060000,87637.99,87356.29,87496.24000,34.820000,-34.820000,2025-12-30 18:30:00,1.963927e+07,9.808946e+06,63776.0


In [290]:
merged_data['volume']=pd.to_numeric(merged_data['volume'])
merged_data['taker_volume']=pd.to_numeric(merged_data['taker_volume'])
merged_data['no_of_trades']=pd.to_numeric(merged_data['no_of_trades'])
merged_data['taker_buy_ratio']=merged_data['taker_volume']/merged_data['volume']
merged_data['avg_trade_size']=merged_data['volume']/merged_data['no_of_trades']
merged_data['vol_z']=(merged_data['volume']-merged_data['volume'].rolling(48).mean())/merged_data['volume'].rolling(48).std()
merged_data

,datetime,open,high,low,close,abs_body,body,open_time,volume,taker_volume,no_of_trades,taker_buy_ratio,avg_trade_size,vol_z
datetime,,,,,,,,,,,,,,
1640995200000000000,2022-01-01 00:00:00,46894.300780,46951.76953,46557.17969,46640.53125,253.769530,-253.769530,2022-01-01 00:00:00,3.275119e+07,1.800036e+07,19674.0,0.549609,1664.694248,NaN
1640997000000000000,2022-01-01 00:30:00,46656.261720,46656.26172,46075,46080.87109,575.390630,-575.390630,2022-01-01 00:30:00,3.712879e+07,1.946179e+07,18934.0,0.524170,1960.958686,NaN
1640998800000000000,2022-01-01 01:00:00,46064.070312,46318.87109,45656.55078,45693.14844,370.921872,-370.921872,2022-01-01 01:00:00,2.754998e+07,1.554390e+07,18930.0,0.564207,1455.360772,NaN
1641000600000000000,2022-01-01 01:30:00,45671.730470,46195.80859,45655.42969,46037.41016,365.679690,365.679690,2022-01-01 01:30:00,1.657717e+07,7.419374e+06,12942.0,0.447566,1280.881525,NaN
1641002400000000000,2022-01-01 02:00:00,46042.019531,46115.21094,45823.94922,45881.53906,160.480471,-160.480471,2022-01-01 02:00:00,1.281610e+07,5.467496e+06,13394.0,0.426611,956.854073,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1767216600000000000,2025-12-31 21:30:00,87830.030000,87932.17,87558,87792.55000,37.480000,-37.480000,2025-12-30 18:30:00,1.963927e+07,9.808946e+06,63776.0,0.499456,307.941394,NaN
1767218400000000000,2025-12-31 22:00:00,87797.910000,87813.02,87259.01,87531.06000,266.850000,-266.850000,2025-12-30 18:30:00,1.963927e+07,9.808946e+06,63776.0,0.499456,307.941394,NaN
1767220200000000000,2025-12-31 22:30:00,87531.060000,87637.99,87356.29,87496.24000,34.820000,-34.820000,2025-12-30 18:30:00,1.963927e+07,9.808946e+06,63776.0,0.499456,307.941394,NaN


In [309]:
merged_data['taker_volume']=pd.to_numeric(merged_data['taker_volume'])

In [315]:
merged_data[70127:70128]['taker_volume'].iloc[0]

9808945.9283699

In [316]:
merged_data[merged_data['taker_volume']==9808945.9283699]

,datetime,open,high,low,close,abs_body,body,open_time,volume,taker_volume,no_of_trades,taker_buy_ratio,avg_trade_size,vol_z
datetime,,,,,,,,,,,,,,
1767119400000000000,2025-12-30 18:30:00,87914.83,88019.99,87794.26,87985.19,70.36,70.36,2025-12-30 18:30:00,1.963927e+07,9.808946e+06,63776.0,0.499456,307.941394,-0.219818
1767121200000000000,2025-12-30 19:00:00,87984.01,88030.92,87808.78,87996.60,12.59,12.59,2025-12-30 18:30:00,1.963927e+07,9.808946e+06,63776.0,0.499456,307.941394,-0.224821
1767123000000000000,2025-12-30 19:30:00,87995.20,88280,87716.6,88032.14,36.94,36.94,2025-12-30 18:30:00,1.963927e+07,9.808946e+06,63776.0,0.499456,307.941394,-0.233822
1767124800000000000,2025-12-30 20:00:00,88032.14,88918.31,87755.89,88767.99,735.85,735.85,2025-12-30 18:30:00,1.963927e+07,9.808946e+06,63776.0,0.499456,307.941394,-0.229295
1767126600000000000,2025-12-30 20:30:00,88768.02,88981,88112.5,88390.06,377.96,-377.96,2025-12-30 18:30:00,1.963927e+07,9.808946e+06,63776.0,0.499456,307.941394,-0.215183
1767128400000000000,2025-12-30 21:00:00,88388.72,89066.24,88335.96,88954.99,566.27,566.27,2025-12-30 18:30:00,1.963927e+07,9.808946e+06,63776.0,0.499456,307.941394,-0.222526
1767130200000000000,2025-12-30 21:30:00,88957.09,89343.52,88664.56,89136.03,178.94,178.94,2025-12-30 18:30:00,1.963927e+07,9.808946e+06,63776.0,0.499456,307.941394,-0.237729
1767132000000000000,2025-12-30 22:00:00,89136.03,89190.74,88484.7,88659.99,476.04,-476.04,2025-12-30 18:30:00,1.963927e+07,9.808946e+06,63776.0,0.499456,307.941394,-0.242533
1767133800000000000,2025-12-30 22:30:00,88660.00,88839.49,88414.06,88549.49,110.51,-110.51,2025-12-30 18:30:00,1.963927e+07,9.808946e+06,63776.0,0.499456,307.941394,-0.251420


In [92]:
with open("logs/30min_performance.csv",mode="a",newline="") as f:
    writer=csv.writer(f)
    for i in range(0,len(data['body']),48):
        row=data['body'][i:i+48]
        writer.writerow(row)

In [93]:
dataset=pd.read_csv('logs/30min_performance.csv')
means=dataset.mean(axis=0)

In [98]:
dataset.loc['mean']=means
dataset

,-253.76952999999776,-575.3906300000017,-370.92187200000626,365.6796899999972,-160.4804709999953,203.5703100000028,232.5976540000047,75.05078000000503,-92.3203199999989,48.16016099999979,...,0.3125,6.5625,-103.51952999999776,97.17186999999831,207.6093800000017,277.1718800000017,-204.5,44.48047000000224,420.91015999999945,-109.42188000000169
0,-78.757810,-193.46094,-54.750000,81.273440,-16.070320,-47.730470,-38.488280,81.589840,47.902340,-2.468750,...,-113.218750,-81.250000,54.69922,72.320320,168.402350,79.089850,370.281250,-161.80859,-241.820310,-471.429680
1,122.269532,-144.69922,23.328130,125.371090,-31.539060,66.941401,-43.000000,133.890620,109.902340,125.089850,...,-165.289060,-76.726560,-212.21094,81.390622,-262.773439,-290.136710,117.878910,208.16015,-105.011710,41.222660
2,-144.992190,150.41797,-238.480470,-574.710940,121.679680,118.929690,-38.429690,286.988280,-38.820310,179.648440,...,77.722650,-65.660150,426.91015,15.621100,-458.417970,-228.519529,76.167970,57.89844,-266.402350,-65.140630
3,-736.671880,427.76172,-139.140629,365.871090,-144.082030,-132.507810,91.878900,-103.730470,36.222650,-114.832030,...,67.312500,-26.250000,324.59766,62.718752,-124.371100,44.089850,-378.070320,71.76954,-360.558600,-159.070310
4,62.660160,-18.06640,-1221.058599,-478.320309,-136.710940,-10.160159,-319.898440,-606.339840,449.113290,219.941410,...,137.988280,-287.167960,-86.76953,78.078130,79.878900,-92.199220,28.089840,173.19922,-31.671875,50.742190
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1456,-35.000000,61.07000,16.030000,-30.860000,12.810000,73.130000,31.210000,12.610000,-122.170000,90.640000,...,50.020000,90.110000,-65.65000,45.250000,-121.440000,-56.320000,34.010000,67.98000,-142.000000,-26.860000
1457,-117.110000,-86.04000,3.270000,58.650000,-68.370000,36.160000,47.040000,31.310000,274.120000,-190.380000,...,101.870000,228.230000,-185.74000,310.770000,-54.940000,-131.990000,89.910000,-205.59000,424.330000,-132.970000
1458,-258.720000,43.35000,-160.240000,-25.920000,-141.460000,92.140000,43.330000,-115.320000,129.110000,-130.020000,...,12.590000,36.940000,735.85000,-377.960000,566.270000,178.940000,-476.040000,-110.51000,-248.600000,7.280000
1459,-78.360000,39.73000,-120.490000,-169.750000,-139.060000,224.540000,136.390000,146.090000,-8.730000,146.780000,...,139.930000,-29.460000,-467.51000,-508.760000,-29.250000,-37.480000,-266.850000,-34.82000,-8.290000,326.370000


In [99]:
with open("logs/30min_performance.csv",mode="a",newline="") as f:
    writer=csv.writer(f)
    writer.writerow(means.values)

In [100]:
with open("logs/30min_performance_abs.csv",mode="a",newline="") as f:
    writer=csv.writer(f)
    for i in range(0,len(data['abs_body']),48):
        row=data['abs_body'][i:i+48]
        writer.writerow(row)

In [101]:
dataset=pd.read_csv('logs/30min_performance_abs.csv')
means=dataset.mean(axis=0)
dataset.loc['mean']=means
with open("logs/30min_performance_abs.csv",mode="a",newline="") as f:
    writer=csv.writer(f)
    writer.writerow(means.values)

In [48]:
data['datetime']=pd.to_datetime(data['datetime'])
data['weekday']=data['datetime'].dt.weekday
data

,datetime,open,high,low,close,abs_body,body,weekday
0,2022-01-01 00:00:00,46894.300780,46951.76953,46557.17969,46640.53125,253.769530,-253.769530,5
1,2022-01-01 00:30:00,46656.261720,46656.26172,46075,46080.87109,575.390630,-575.390630,5
2,2022-01-01 01:00:00,46064.070312,46318.87109,45656.55078,45693.14844,370.921872,-370.921872,5
3,2022-01-01 01:30:00,45671.730470,46195.80859,45655.42969,46037.41016,365.679690,365.679690,5
4,2022-01-01 02:00:00,46042.019531,46115.21094,45823.94922,45881.53906,160.480471,-160.480471,5
...,...,...,...,...,...,...,...,...
70123,2025-12-31 21:30:00,87830.030000,87932.17,87558,87792.55000,37.480000,-37.480000,2
70124,2025-12-31 22:00:00,87797.910000,87813.02,87259.01,87531.06000,266.850000,-266.850000,2
70125,2025-12-31 22:30:00,87531.060000,87637.99,87356.29,87496.24000,34.820000,-34.820000,2
70126,2025-12-31 23:00:00,87496.250000,87550.34,87400.76,87487.96000,8.290000,-8.290000,2


In [50]:
grouped=data.groupby("weekday")
grouped["body"].mean()

weekday
0    0.016129
1    0.671313
2    4.202240
3   -2.347562
4   -1.366429
5   -0.380943
6    2.083995
Name: body, dtype: float64

In [52]:
grouped["abs_body"].mean()

weekday
0    152.870233
1    152.076369
2    145.616293
3    149.039251
4    155.200781
5     97.148329
6     90.359163
Name: abs_body, dtype: float64

In [53]:
data['month']=data['datetime'].dt.month
data

,datetime,open,high,low,close,abs_body,body,weekday,month
0,2022-01-01 00:00:00,46894.300780,46951.76953,46557.17969,46640.53125,253.769530,-253.769530,5,1
1,2022-01-01 00:30:00,46656.261720,46656.26172,46075,46080.87109,575.390630,-575.390630,5,1
2,2022-01-01 01:00:00,46064.070312,46318.87109,45656.55078,45693.14844,370.921872,-370.921872,5,1
3,2022-01-01 01:30:00,45671.730470,46195.80859,45655.42969,46037.41016,365.679690,365.679690,5,1
4,2022-01-01 02:00:00,46042.019531,46115.21094,45823.94922,45881.53906,160.480471,-160.480471,5,1
...,...,...,...,...,...,...,...,...,...
70123,2025-12-31 21:30:00,87830.030000,87932.17,87558,87792.55000,37.480000,-37.480000,2,12
70124,2025-12-31 22:00:00,87797.910000,87813.02,87259.01,87531.06000,266.850000,-266.850000,2,12
70125,2025-12-31 22:30:00,87531.060000,87637.99,87356.29,87496.24000,34.820000,-34.820000,2,12
70126,2025-12-31 23:00:00,87496.250000,87550.34,87400.76,87487.96000,8.290000,-8.290000,2,12


In [54]:
grouped1=data.groupby("month")
grouped1["body"].mean()

month
1     1.500843
2    -0.045879
3     0.555033
4    -0.800682
5     1.628302
6    -2.226954
7     3.279645
8    -3.528276
9     1.687959
10    1.706564
11    1.433070
12   -0.308658
Name: body, dtype: float64

In [56]:
grouped1["abs_body"].mean()

month
1     141.991350
2     136.668648
3     184.570013
4     142.520338
5     133.575085
6     115.098166
7     117.601697
8     119.068575
9      97.721774
10    122.790721
11    159.077386
12    143.829659
Name: abs_body, dtype: float64

In [293]:
with open("logs/30min_performance_takerbuyratio.csv",mode="a",newline="") as f:
    writer=csv.writer(f)
    for i in range(0,len(merged_data['taker_buy_ratio']),48):
        row=merged_data['taker_buy_ratio'][i:i+48]
        writer.writerow(row)

In [294]:
dataset=pd.read_csv('logs/30min_performance_takerbuyratio.csv')
means=dataset.mean(axis=0)
dataset.loc['mean']=means
with open("logs/30min_performance_takerbuyratio.csv",mode="a",newline="") as f:
    writer=csv.writer(f)
    writer.writerow(means.values*100)

In [317]:
with open("logs/30min_performance_avgtradesize.csv",mode="a",newline="") as f:
    writer=csv.writer(f)
    for i in range(0,len(merged_data['avg_trade_size']),48):
        row=merged_data['avg_trade_size'][i:i+48]
        writer.writerow(row)
dataset=pd.read_csv('logs/30min_performance_avgtradesize.csv')
means=dataset.mean(axis=0)
dataset.loc['mean']=means
with open("logs/30min_performance_avgtradesize.csv",mode="a",newline="") as f:
    writer=csv.writer(f)
    writer.writerow(means.values)

In [318]:
with open("logs/30min_performance_volz.csv",mode="a",newline="") as f:
    writer=csv.writer(f)
    for i in range(0,len(merged_data['vol_z']),48):
        row=merged_data['vol_z'][i:i+48]
        writer.writerow(row)
dataset=pd.read_csv('logs/30min_performance_volz.csv')
means=dataset.mean(axis=0)
dataset.loc['mean']=means
with open("logs/30min_performance_volz.csv",mode="a",newline="") as f:
    writer=csv.writer(f)
    writer.writerow(means.values)